In [4]:
import ee
import geemap

geemap.ee_initialize()

Enter verification code:  4/1ASc3gC3Ymnhkl5LmwGY85iKDI8H3XxETYs6gkrlnCnWkGTzspmX5yXq5ILE



Successfully saved authorization token.


In [5]:
# quick sanity check (should print a CRS string like 'EPSG:4326' or similar)
img = ee.Image("USGS/SRTMGL1_003")
print(img.projection().crs().getInfo())

EPSG:4326


In [6]:
import numpy as np
import pandas as pd

print("numpy:", np.__version__)
print("pandas:", pd.__version__)

inventory = pd.DataFrame(
    [
        {
            "source": "USGS EarthExplorer",
            "dataset": "Landsat C2 L2 (SR)",
            "resolution": "30 m",
            "license": "check metadata",
            "notes": "downloaded 1 scene",
        },
        {
            "source": "GEE (via geemap)",
            "dataset": "Sentinel-2 SR Harmonized",
            "resolution": "10-20 m",
            "license": "check collection docs",
            "notes": "NDVI + time series",
        },
    ]
)

inventory

numpy: 2.4.1
pandas: 3.0.0


,source,dataset,resolution,license,notes
0,USGS EarthExplorer,Landsat C2 L2 (SR),30 m,check metadata,downloaded 1 scene
1,GEE (via geemap),Sentinel-2 SR Harmonized,10-20 m,check collection docs,NDVI + time series


In [12]:
import ee
import geemap
from geemap import chart

geemap.ee_initialize()

# 1) ROI (edit if you want)
roi = ee.Geometry.Rectangle([126.76, 37.42, 127.19, 37.70])

# 2) Map
m = geemap.Map(center=[37.56, 126.975], zoom=10)
m.add_basemap("HYBRID")
m.addLayer(roi, {"color": "yellow"}, "ROI", False)

# 3) Sentinel-2 SR (cloud-masked) with timestamp preserved
def mask_s2_sr(img):
    scl = img.select("SCL")
    mask = (scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11)))
    out = img.updateMask(mask).divide(10000)
    return out.copyProperties(img, img.propertyNames())

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2026-01-01", "2026-01-31")
    .map(mask_s2_sr)
    .filter(ee.Filter.notNull(["system:time_start"])))

s2_median = s2.median().clip(roi)
m.addLayer(s2_median, {"min": 0, "max": 0.3, "bands": ["B4","B3","B2"]}, "Sentinel-2 RGB")

# 4) NDVI
ndvi = s2_median.normalizedDifference(["B8","B4"]).rename("NDVI")
ndvi_vis = {"min": 0, "max": 1, "palette": ["#440154","#3b528b","#21918c","#5ec962","#fde725"]}
m.addLayer(ndvi, ndvi_vis, "NDVI")
m.add_colorbar(ndvi_vis, label="NDVI", layer_name="NDVI")

m

Map(center=[37.56, 126.975], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [13]:
geemap.ee_export_image_to_drive(
    image=ndvi,
    description='ndvi_demo_geovis_Seoul',
    folder='geemap_exports',
    region=roi,
    scale=20,
    maxPixels=1e13)

print('Export task submitted. Check the Earth Engine Task tab.')

Export task submitted. Check the Earth Engine Task tab.
